# Lesson 20 | First MaleCNS subset load: verify the image before trusting it

The project eventually needs the same converted network image to be consumed by software and FPGA.

Today asks one question:

> **How can we prove that two consumers received the same versioned network bytes?**

Primary new concept: **manifest + checksum as an integrity contract.**

Important boundary: the formal RMD-017/018 MaleCNS artifact is not yet declared complete. This lesson uses a tiny teaching fixture to learn the loading contract; it does **not** claim that the fixture is real MaleCNS data.

## 1. Concept ledger

**Already known:** connectome nodes/edges/metadata, binary data movement, host/FPGA boundaries.

**New today:** **manifest** (a small description of an artifact) and **checksum** (a deterministic digest of exact bytes).

**Preview only:** real MaleCNS conversion, formal binary schema, 1K differential test, and full dataset loading.

## 2. What must travel together?

A reproducible network artifact needs more than a file name. At minimum we want:

- schema/version identifier;
- byte length;
- checksum of the exact image bytes;
- enough provenance to know which converter/data release produced it.

A checksum answers **“are these bytes identical?”** It does not answer **“is this scientifically the correct network?”**

## 3. Conversion and replay picture

```mermaid
flowchart LR
  A["source connectome data"] --> B["versioned converter"]
  B --> C["binary image"]
  B --> D["manifest + checksum"]
  C --> E["software consumer"]
  C --> F["FPGA consumer"]
  D --> E
  D --> F
```

## 4. Run: make and verify a teaching image

The payload below is just a byte fixture chosen so the integrity workflow is runnable without network access or a formal MaleCNS release artifact.

In [ ]:
import hashlib
import json

payload = bytes([19, 20, 1, 0, 3, 2, 7, 11])
manifest = {
    "schema_version": "teaching-v1",
    "byte_count": len(payload),
    "sha256": hashlib.sha256(payload).hexdigest(),
}

software_bytes = bytes(payload)
fpga_replay_bytes = bytes(payload)

print(json.dumps(manifest, indent=2))
print("software checksum:", hashlib.sha256(software_bytes).hexdigest())
print("FPGA replay checksum:", hashlib.sha256(fpga_replay_bytes).hexdigest())
print("same image:", software_bytes == fpga_replay_bytes)

## 5. Observe

Both consumers receive identical bytes and therefore the same SHA-256 checksum.

This is an **integrity proof**, not yet a neural correctness proof. T-015 later compares the behavior produced by the real subset in software and FPGA.

## 6. Why version the converter?

If the conversion rule changes — for example edge ordering, weight encoding, or record width — the same scientific source dataset may produce different binary bytes.

That is why the converter/schema version belongs in the manifest. Reproducibility needs the **data version and conversion rule**, not only the source name.

## 7. Differential test comes after integrity

A useful order is:

1. verify manifest and checksum;
2. load the same image into both implementations;
3. replay the same initial state and input events;
4. compare outputs/state using the approved oracle.

If step 1 fails, behavioral differences are impossible to interpret cleanly.

## 8. Try It

Change one payload byte, recompute the digest, and predict which manifest fields change.

Does a different checksum tell you *why* the byte changed?

## 9. Exercise

[Lesson 20 exercise: build a minimal image manifest](../../exercises/en/20_load_malecns_subset.ipynb)

## 10. AI Task

Ask an AI for a proposed manifest schema. Mark each field as integrity, provenance, schema/version, or experiment state. Reject fields that silently mix runtime neuron state into the static connectome image.

## 11. Human Check

Explain why equal checksums are necessary but not sufficient for a correct neural result. Why must software and FPGA differential tests consume the exact same binary image?

## 12. Engineering Handoff

Maps to `RMD-017 / RMD-018`, `MOD-011`, and `T-014 / T-015`. The formal engineering slice must replace the teaching fixture with a documented MaleCNS-derived artifact before claiming a real-subset result.

## 13. Project Trace

- Lesson: `LSN-020`
- Mapping: `RMD-018` with prerequisite `RMD-017`
- Requirement path: `TRACE-C-001`
- Integrity oracle: `T-014`
- Real-subset behavioral oracle: `T-015`

## 14. Exit Ticket

Given a binary image and manifest, you can verify byte count and checksum, explain what that proves, and state what still requires a differential behavioral test.